# Rappi Restaurant Partners — Business Standing Analysis

**Objective:** Understand where the business stands right now across the restaurant partner portfolio.  
Key questions:
- What is the overall health of our restaurant portfolio?
- How are KPIs distributed (ratings, cancellations, delivery time, orders)?
- Which cities and verticals are performing best/worst?
- Which restaurants need immediate attention?
- How are KAMs managing their portfolios?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.dpi"] = 120

df = pd.read_csv("dataset.csv", encoding="utf-8-sig")
print(f"Dataset: {df.shape[0]} restaurants × {df.shape[1]} columns")
df.head()

In [ ]:
df.info()
print("\n")
df.describe()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing[missing > 0] if missing.any() else print("No missing values ✓")

---
## 1. Portfolio Health Overview
How healthy is the overall restaurant portfolio right now?

In [ ]:
# Clean semaforo labels for easier handling
df["riesgo"] = df["semaforo_riesgo"].str.extract(r"(ESTABLE|EN RIESGO|CRÍTICO)")

risk_counts = df["riesgo"].value_counts()
risk_pct = (risk_counts / len(df) * 100).round(1)

color_map = {"ESTABLE": "#2ecc71", "EN RIESGO": "#f39c12", "CRÍTICO": "#e74c3c"}
order = ["ESTABLE", "EN RIESGO", "CRÍTICO"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Pie chart
axes[0].pie(
    [risk_counts[r] for r in order],
    labels=[f"{r}\n{risk_counts[r]} ({risk_pct[r]}%)" for r in order],
    colors=[color_map[r] for r in order],
    startangle=90, wedgeprops=dict(width=0.45, edgecolor="white"),
    textprops={"fontsize": 11}
)
axes[0].set_title("Risk Traffic Light Distribution", fontsize=13, fontweight="bold")

# Bar chart
bars = axes[1].bar(order, [risk_counts[r] for r in order], color=[color_map[r] for r in order], edgecolor="white", width=0.55)
for bar, r in zip(bars, order):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f"{risk_counts[r]}", ha="center", fontweight="bold")
axes[1].set_ylabel("# Restaurants")
axes[1].set_title("Restaurants by Risk Level", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

print(f"\nSummary: {risk_pct.get('ESTABLE', 0)}% stable, {risk_pct.get('EN RIESGO', 0)}% at risk, {risk_pct.get('CRÍTICO', 0)}% critical.")

---
## 2. KPI Snapshot — Where Do We Stand?
High-level view of the key operational metrics across the entire portfolio.

In [ ]:
kpis = {
    "Avg Rating": df["rating_actual"].mean(),
    "Median Rating": df["rating_actual"].median(),
    "Avg Cancellation %": df["tasa_cancelacion_pct"].mean(),
    "Avg Delivery Time (min)": df["tiempo_entrega_avg_min"].mean(),
    "Total Orders (7d)": df["ordenes_7d"].sum(),
    "Avg Order Growth %": df["var_ordenes_pct"].mean(),
    "Total Complaints (7d)": df["quejas_7d"].sum(),
    "Avg NPS": df["nps_score"].mean(),
    "Avg Ticket (MXN)": df["valor_ticket_prom_mxn"].mean(),
}

kpi_df = pd.DataFrame.from_dict(kpis, orient="index", columns=["Value"])
kpi_df["Value"] = kpi_df["Value"].map(lambda x: f"{x:,.1f}")
kpi_df

In [ ]:
# Distribution of key metrics
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

metrics = [
    ("rating_actual", "Current Rating", "#3498db"),
    ("tasa_cancelacion_pct", "Cancellation Rate (%)", "#e74c3c"),
    ("tiempo_entrega_avg_min", "Delivery Time (min)", "#f39c12"),
    ("ordenes_7d", "Orders (Last 7d)", "#2ecc71"),
    ("nps_score", "NPS Score", "#9b59b6"),
    ("valor_ticket_prom_mxn", "Avg Ticket (MXN)", "#1abc9c"),
]

for ax, (col, title, color) in zip(axes.flat, metrics):
    ax.hist(df[col], bins=20, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(df[col].mean(), color="black", linestyle="--", linewidth=1.2, label=f"Mean: {df[col].mean():.1f}")
    ax.axvline(df[col].median(), color="gray", linestyle=":", linewidth=1.2, label=f"Median: {df[col].median():.1f}")
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=8)

plt.suptitle("KPI Distributions Across All Restaurants", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 3. KPIs by Risk Level
How do key metrics differ between stable, at-risk, and critical restaurants?

In [ ]:
risk_summary = df.groupby("riesgo").agg(
    count=("restaurant_id", "count"),
    avg_rating=("rating_actual", "mean"),
    avg_cancel_pct=("tasa_cancelacion_pct", "mean"),
    avg_delivery_min=("tiempo_entrega_avg_min", "mean"),
    avg_orders_7d=("ordenes_7d", "mean"),
    avg_order_growth=("var_ordenes_pct", "mean"),
    avg_complaints_7d=("quejas_7d", "mean"),
    avg_nps=("nps_score", "mean"),
    avg_ticket=("valor_ticket_prom_mxn", "mean"),
).reindex(["ESTABLE", "EN RIESGO", "CRÍTICO"]).round(1)

risk_summary

In [ ]:
# Box plots: key metrics split by risk level
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

box_metrics = [
    ("rating_actual", "Current Rating"),
    ("tasa_cancelacion_pct", "Cancellation Rate (%)"),
    ("tiempo_entrega_avg_min", "Delivery Time (min)"),
    ("var_ordenes_pct", "Order Growth (%)"),
    ("nps_score", "NPS Score"),
    ("quejas_7d", "Complaints (7d)"),
]

palette = {"ESTABLE": "#2ecc71", "EN RIESGO": "#f39c12", "CRÍTICO": "#e74c3c"}

for ax, (col, title) in zip(axes.flat, box_metrics):
    sns.boxplot(data=df, x="riesgo", y=col, order=order, palette=palette, ax=ax, fliersize=3)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("")

plt.suptitle("KPI Comparison by Risk Level", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 4. City-Level Breakdown
Which cities are performing well and which need attention?

In [ ]:
city_summary = df.groupby("ciudad").agg(
    restaurants=("restaurant_id", "count"),
    avg_rating=("rating_actual", "mean"),
    avg_cancel_pct=("tasa_cancelacion_pct", "mean"),
    avg_delivery_min=("tiempo_entrega_avg_min", "mean"),
    total_orders_7d=("ordenes_7d", "sum"),
    avg_order_growth=("var_ordenes_pct", "mean"),
    avg_nps=("nps_score", "mean"),
    pct_critical=("riesgo", lambda x: (x == "CRÍTICO").mean() * 100),
).round(1).sort_values("total_orders_7d", ascending=False)

city_summary

In [ ]:
# Risk distribution by city
city_risk = pd.crosstab(df["ciudad"], df["riesgo"])[order]
city_risk_pct = city_risk.div(city_risk.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Stacked bar (absolute)
city_risk.plot(kind="barh", stacked=True, color=[color_map[r] for r in order], ax=axes[0], edgecolor="white")
axes[0].set_title("Risk Distribution by City (Count)", fontweight="bold")
axes[0].set_xlabel("# Restaurants")
axes[0].legend(title="Risk Level")

# Stacked bar (percentage)
city_risk_pct.plot(kind="barh", stacked=True, color=[color_map[r] for r in order], ax=axes[1], edgecolor="white")
axes[1].set_title("Risk Distribution by City (%)", fontweight="bold")
axes[1].set_xlabel("% of Restaurants")
axes[1].legend(title="Risk Level")
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.show()

---
## 5. Vertical Breakdown
How is each business vertical (Comida, Farmacia, Bebidas, Mercado) performing?

In [ ]:
vertical_summary = df.groupby("vertical").agg(
    restaurants=("restaurant_id", "count"),
    avg_rating=("rating_actual", "mean"),
    avg_cancel_pct=("tasa_cancelacion_pct", "mean"),
    avg_delivery_min=("tiempo_entrega_avg_min", "mean"),
    total_orders_7d=("ordenes_7d", "sum"),
    avg_order_growth=("var_ordenes_pct", "mean"),
    avg_nps=("nps_score", "mean"),
    avg_ticket=("valor_ticket_prom_mxn", "mean"),
    pct_critical=("riesgo", lambda x: (x == "CRÍTICO").mean() * 100),
).round(1).sort_values("total_orders_7d", ascending=False)

vertical_summary

In [ ]:
# Vertical risk distribution + avg rating
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vert_risk = pd.crosstab(df["vertical"], df["riesgo"])[order]
vert_risk.plot(kind="bar", stacked=True, color=[color_map[r] for r in order], ax=axes[0], edgecolor="white")
axes[0].set_title("Risk Distribution by Vertical", fontweight="bold")
axes[0].set_ylabel("# Restaurants")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend(title="Risk Level")

# Avg rating & NPS by vertical
vert_metrics = df.groupby("vertical")[["rating_actual", "nps_score"]].mean().sort_values("rating_actual", ascending=False)
vert_metrics.plot(kind="bar", ax=axes[1], color=["#3498db", "#9b59b6"], edgecolor="white")
axes[1].set_title("Avg Rating & NPS by Vertical", fontweight="bold")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(["Rating (left scale)", "NPS"])

plt.tight_layout()
plt.show()

---
## 6. Critical Restaurants — Immediate Attention List
Which restaurants are flagged as critical and what are their key pain points?

In [ ]:
critical = df[df["riesgo"] == "CRÍTICO"].sort_values("rating_actual")

critical_display = critical[[
    "restaurant_id", "nombre", "ciudad", "vertical", "rating_actual",
    "delta_rating", "tasa_cancelacion_pct", "tiempo_entrega_avg_min",
    "ordenes_7d", "var_ordenes_pct", "quejas_7d", "nps_score", "kam_asignado"
]].reset_index(drop=True)

print(f"🔴 {len(critical)} critical restaurants requiring immediate attention:\n")
critical_display

In [ ]:
# What makes critical restaurants different? Scatter: cancellation rate vs rating colored by risk
fig, ax = plt.subplots(figsize=(10, 6))

for risk in order:
    subset = df[df["riesgo"] == risk]
    ax.scatter(
        subset["tasa_cancelacion_pct"], subset["rating_actual"],
        c=color_map[risk], label=risk, alpha=0.7, s=subset["ordenes_7d"] * 0.4,
        edgecolors="white", linewidth=0.5
    )

ax.set_xlabel("Cancellation Rate (%)", fontsize=11)
ax.set_ylabel("Current Rating", fontsize=11)
ax.set_title("Rating vs Cancellation Rate (bubble size = weekly orders)", fontsize=13, fontweight="bold")
ax.legend(title="Risk Level")
plt.tight_layout()
plt.show()

---
## 7. KAM Portfolio Performance
How are the Key Account Managers performing across their restaurant portfolios?

In [ ]:
kam_summary = df.groupby("kam_asignado").agg(
    restaurants=("restaurant_id", "count"),
    avg_rating=("rating_actual", "mean"),
    total_orders_7d=("ordenes_7d", "sum"),
    avg_order_growth=("var_ordenes_pct", "mean"),
    avg_cancel_pct=("tasa_cancelacion_pct", "mean"),
    avg_nps=("nps_score", "mean"),
    total_complaints=("quejas_7d", "sum"),
    n_critical=("riesgo", lambda x: (x == "CRÍTICO").sum()),
    n_at_risk=("riesgo", lambda x: (x == "EN RIESGO").sum()),
    n_stable=("riesgo", lambda x: (x == "ESTABLE").sum()),
).round(1).sort_values("avg_rating", ascending=False)

kam_summary

In [ ]:
# KAM risk composition
fig, ax = plt.subplots(figsize=(12, 5))

kam_risk = pd.crosstab(df["kam_asignado"], df["riesgo"])[order]
kam_risk.plot(kind="barh", stacked=True, color=[color_map[r] for r in order], ax=ax, edgecolor="white")
ax.set_title("KAM Portfolio — Risk Composition", fontsize=13, fontweight="bold")
ax.set_xlabel("# Restaurants")
ax.set_ylabel("")
ax.legend(title="Risk Level")
plt.tight_layout()
plt.show()

---
## 8. Order Volume & Growth Trends
Which restaurants are growing vs declining? Is the overall portfolio trending up or down?

In [ ]:
# Overall order volume change
total_orders_current = df["ordenes_7d"].sum()
total_orders_previous = df["ordenes_7d_anterior"].sum()
overall_growth = (total_orders_current - total_orders_previous) / total_orders_previous * 100

growing = (df["var_ordenes_pct"] > 0).sum()
declining = (df["var_ordenes_pct"] < 0).sum()
flat = (df["var_ordenes_pct"] == 0).sum()

print(f"Total orders this week:  {total_orders_current:,}")
print(f"Total orders last week:  {total_orders_previous:,}")
print(f"Overall WoW growth:      {overall_growth:+.1f}%")
print(f"\nGrowing restaurants:  {growing} ({growing/len(df)*100:.0f}%)")
print(f"Declining restaurants: {declining} ({declining/len(df)*100:.0f}%)")
print(f"Flat restaurants:     {flat}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of order growth
axes[0].hist(df["var_ordenes_pct"], bins=25, color="#3498db", edgecolor="white", alpha=0.85)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.5)
axes[0].axvline(df["var_ordenes_pct"].mean(), color="black", linestyle="--", linewidth=1, label=f"Mean: {df['var_ordenes_pct'].mean():.1f}%")
axes[0].set_title("Distribution of WoW Order Growth (%)", fontweight="bold")
axes[0].set_xlabel("Order Growth (%)")
axes[0].legend()

# Top 15 growers vs top 15 decliners
top_grow = df.nlargest(10, "var_ordenes_pct")[["nombre", "var_ordenes_pct", "riesgo"]]
top_decline = df.nsmallest(10, "var_ordenes_pct")[["nombre", "var_ordenes_pct", "riesgo"]]
combined = pd.concat([top_grow, top_decline]).sort_values("var_ordenes_pct")

colors = [color_map.get(r, "#95a5a6") for r in combined["riesgo"]]
axes[1].barh(combined["nombre"], combined["var_ordenes_pct"], color=colors, edgecolor="white")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Top 10 Growers & Decliners (WoW %)", fontweight="bold")
axes[1].set_xlabel("Order Growth (%)")

plt.tight_layout()
plt.show()

---
## 9. Correlation Analysis
How do the key metrics relate to each other?

In [ ]:
corr_cols = [
    "rating_actual", "delta_rating", "tasa_cancelacion_pct",
    "tiempo_entrega_avg_min", "ordenes_7d", "var_ordenes_pct",
    "quejas_7d", "nps_score", "valor_ticket_prom_mxn"
]

corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = [[False]*len(corr_cols) for _ in range(len(corr_cols))]
for i in range(len(corr_cols)):
    for j in range(i):
        mask[i][j] = True  # we actually want upper triangle hidden

sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title("Correlation Matrix — Key Operational Metrics", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 10. Top & Bottom Performers
Quick reference: the best and worst restaurants across key metrics.

In [ ]:
show_cols = ["nombre", "ciudad", "vertical", "rating_actual", "ordenes_7d", "tasa_cancelacion_pct", "nps_score", "riesgo"]

print("⭐ Top 10 by Rating")
display(df.nlargest(10, "rating_actual")[show_cols].reset_index(drop=True))

print("\n⚠️ Bottom 10 by Rating")
display(df.nsmallest(10, "rating_actual")[show_cols].reset_index(drop=True))

print("\n📦 Top 10 by Order Volume (7d)")
display(df.nlargest(10, "ordenes_7d")[show_cols].reset_index(drop=True))

print("\n🚨 Top 10 by Cancellation Rate")
display(df.nlargest(10, "tasa_cancelacion_pct")[show_cols + ["quejas_7d"]].reset_index(drop=True))

---
## Summary

This notebook provides a current-state snapshot of the Rappi restaurant partner portfolio. Key areas to investigate further:

1. **Critical restaurants** — deep-dive into root causes (delivery issues? quality? cancellations?) and assign action plans per KAM
2. **City-specific strategies** — cities with high critical % may need market-level interventions
3. **Vertical performance gaps** — understand if underperformance is structural or operational
4. **KAM effectiveness** — identify best practices from top-performing KAMs and replicate
5. **Growth vs decline patterns** — understand what drives the top growers and stabilize decliners